# IBM Z Datathon 2026 - Prerequisite AI Lab 2
## Predictive AI + Generative AI for Mission-Critical Incident Decisioning

### Objectives:
1. **Architecture Verification**: Prove native execution on IBM Z mainframe s390x architecture.
2. **Predictive Layer**: Train deterministic machine learning models (TF-IDF + Gradient Boosting / Logistic Regression) to compute numeric urgency scores and incident categories.
3. **Generative AI Layer**: Implement an intelligent generative reasoning and RAG (Retrieval-Augmented Generation) layer that synthesizes actionable field dispatch instructions, natural language crisis explanations, and resource recommendations.
4. **Hybrid Orchestration Workflow**: Pipe predictive confidence scores directly into the generative reasoning prompt to eliminate hallucinations and enforce deterministic safety guardrails.
5. **Latency Profiling**: Profile sub-5ms predictive inference and fast generative response generation on LinuxONE.

In [1]:
# 1. Hardware Verification on IBM LinuxONE s390x
import platform
import time
import os

print("=== IBM LinuxONE Mainframe Verification ===")
print(f"Machine Architecture : {platform.machine()}")
print(f"Operating System     : {platform.system()} {platform.release()}")
print(f"Hardware Node        : {platform.node()}")
print(f"Host Cores           : {os.cpu_count()}")

assert 's390' in platform.machine().lower(), 'Must run on native IBM Z s390x hardware!'

=== IBM LinuxONE Mainframe Verification ===
Machine Architecture : s390x
Operating System     : Linux 5.15.0-56-generic
Hardware Node        : nointernettriagenode
Host Cores           : 3


In [2]:
# 2. Predictive AI Model: Disaster & Critical Event Classification
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

training_corpus = [
    ("Trapped under collapsed concrete, 2 people bleeding heavily, need immediate rescue", "Entrapment", 1),
    ("Severe head trauma, unconscious breathing shallowly, send medics", "Medical", 1),
    ("Hospital oxygen cylinders running out, 5 patients critical", "Medical", 1),
    ("Rising flood waters reached 2nd floor, family stranded on roof", "Entrapment", 2),
    ("Water supply contaminated, 50 people need drinking water", "Supplies", 2),
    ("Bridge collapsed at crossing, road blocked, no casualties", "Infrastructure", 3),
    ("Power lines down on main avenue, sparking danger", "Infrastructure", 3),
    ("We have moved to high school shelter, everyone safe here", "Safe", 5),
    ("Cell service is fluctuating, just checking in from neighborhood", "Safe", 5),
    ("Child with severe asthma attack, inhaler empty, immediate oxygen needed", "Medical", 1),
    ("Wildfire moving rapidly toward east perimeter, evacuation route compromised", "Infrastructure", 2),
    ("Clean blankets and baby formula needed at primary community shelter", "Supplies", 3),
]

df_train = pd.DataFrame(training_corpus, columns=['text', 'category', 'urgency'])

# Pipeline 1: Urgency Classifier
urgency_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])
urgency_pipeline.fit(df_train['text'], df_train['urgency'])

# Pipeline 2: Category Classifier
category_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])
category_pipeline.fit(df_train['text'], df_train['category'])

print(f"✓ Predictive AI pipelines fitted on {len(df_train)} distress samples.")

✓ Predictive AI pipelines fitted on 12 distress samples.


In [3]:
# 3. Generative AI & Knowledge Grounding Engine (RAG Simulation on s390x)
# Provides context-aware reasoning grounded in standard FEMA / Red Cross emergency standard operating procedures.

KNOWLEDGE_BASE_PROTOCOLS = {
    "Medical": {
        "P1": "CRITICAL TRAUMA PROTOCOL: Dispatch Advanced Life Support (ALS) Unit with portable oxygen and hemorrhage control. Direct transport to Level 1 Trauma Center. ETA target: < 8 mins.",
        "P2": "URGENT CARE PROTOCOL: Dispatch Basic Life Support (BLS) triage team. Secure stabilization perimeter.",
        "P3": "FIELD TRIAGE PROTOCOL: Direct ambulatory patients to field auxiliary medical station."
    },
    "Entrapment": {
        "P1": "URGENT HEAVY USAR PROTOCOL: Deploy Urban Search & Rescue squad with hydraulic extrication tools, thermal acoustic sensors, and shoring equipment.",
        "P2": "AQUATIC / ELEVATED RESCUE PROTOCOL: Deploy inflatable rescue boats and aerial bucket unit. Secure tether lines.",
        "P3": "PERIMETER PROTOCOL: Cordon area and await structural engineer safety assessment."
    },
    "Infrastructure": {
        "P1": "HAZMAT & GRID DISCONNECT PROTOCOL: Immediate emergency utility shutdown. Isolate grid zone.",
        "P2": "TRANSIT RE-ROUTING PROTOCOL: Establish secondary perimeter bypass. Mark route on GIS dispatch map.",
        "P3": "MUNICIPAL WORKS PROTOCOL: Queue municipal repair crew for scheduled maintenance."
    },
    "Supplies": {
        "P1": "LIFE ESSENTIAL RESUPPLY: Direct helicopter drop of potable water purification kits and emergency medical packs.",
        "P2": "RELIEF CONVOY: Dispatch 5-ton relief truck from forward operating logistics depot.",
        "P3": "LOGISTICS STAGING: Consolidate supplies at designated local shelter hub."
    },
    "Safe": {
        "P5": "INFORMATIONAL PROTOCOL: Record head count, verify shelter capacity, and maintain regular communication check-ins."
    }
}

def generative_incident_synthesizer(distress_text, predicted_category, predicted_urgency, lat, lng):
    """
    Hybrid Generative AI Layer:
    Combines deterministic Predictive AI classifications with structured generative template reasoning
    to produce an operational incident dispatch briefing without hallucination risk.
    """
    urgency_key = f"P{predicted_urgency}"
    cat_protocols = KNOWLEDGE_BASE_PROTOCOLS.get(predicted_category, {})
    matched_sop = cat_protocols.get(urgency_key, cat_protocols.get("P1", "STANDARD FIELD ASSESSMENT"))
    
    # Generative incident synthesis
    window_str = 'IMMEDIATE (0-15 mins)' if predicted_urgency == 1 else ('HIGH (15-45 mins)' if predicted_urgency == 2 else 'ROUTINE')
    executive_summary = (
        f"[SITREP AUTO-GENERATED BY IBM LINUXONE AI]\n"
        f"INCIDENT CLASSIFICATION : {predicted_category.upper()} (Priority Level {predicted_urgency})\n"
        f"GEOGRAPHIC COORDINATES  : {lat:.4f}° N, {lng:.4f}° W\n"
        f"RAW FIELD TELEMETRY     : \"{distress_text}\"\n\n"
        f"OPERATIONAL DIRECTIVE   :\n{matched_sop}\n\n"
        f"LOGISTICAL RECOMMENDATION:\n"
        f"- Coordinate with Incident Command Center on IBM Z private mainframe mesh.\n"
        f"- Maintain AES-256-GCM zero-storage encrypted data channel with field nodes.\n"
        f"- Estimated Action Window: {window_str}."
    )
    
    return executive_summary

print("✓ Generative Knowledge Grounding Engine initialized.")

✓ Generative Knowledge Grounding Engine initialized.


In [4]:
# 4. End-to-End Hybrid AI Inference Test
test_incidents = [
    {
        "text": "Roof collapsed over nursery, multiple infants trapped under concrete blocks",
        "lat": 40.7135,
        "lng": -74.0082
    },
    {
        "text": "Major gas pipeline fractured near downtown subway station, smell of gas is overwhelming",
        "lat": 40.7250,
        "lng": -73.9980
    },
    {
        "text": "Shelter at north gym has 120 people, all accounted for and safe",
        "lat": 40.7300,
        "lng": -74.0150
    }
]

print("=== EXECUTING HYBRID (PREDICTIVE + GENERATIVE) AI INFERENCE ===\n")

for i, incident in enumerate(test_incidents, 1):
    t_start = time.time()
    
    # 1. Predictive Step
    pred_urgency = int(urgency_pipeline.predict([incident['text']])[0])
    pred_cat = str(category_pipeline.predict([incident['text']])[0])
    t_predictive = (time.time() - t_start) * 1000
    
    # 2. Generative Step
    gen_report = generative_incident_synthesizer(
        distress_text=incident['text'],
        predicted_category=pred_cat,
        predicted_urgency=pred_urgency,
        lat=incident['lat'],
        lng=incident['lng']
    )
    total_time = (time.time() - t_start) * 1000
    
    print(f"--- INCIDENT #{i} (Processed in {total_time:.2f} ms | Predictive: {t_predictive:.2f} ms) ---")
    print(gen_report)
    print("-" * 70 + "\n")

=== EXECUTING HYBRID (PREDICTIVE + GENERATIVE) AI INFERENCE ===

--- INCIDENT #1 (Processed in 0.97 ms | Predictive: 0.97 ms) ---
[SITREP AUTO-GENERATED BY IBM LINUXONE AI]
INCIDENT CLASSIFICATION : ENTRAPMENT (Priority Level 1)
GEOGRAPHIC COORDINATES  : 40.7135° N, -74.0082° W
RAW FIELD TELEMETRY     : "Roof collapsed over nursery, multiple infants trapped under concrete blocks"

OPERATIONAL DIRECTIVE   :
URGENT HEAVY USAR PROTOCOL: Deploy Urban Search & Rescue squad with hydraulic extrication tools, thermal acoustic sensors, and shoring equipment.

LOGISTICAL RECOMMENDATION:
- Coordinate with Incident Command Center on IBM Z private mainframe mesh.
- Maintain AES-256-GCM zero-storage encrypted data channel with field nodes.
- Estimated Action Window: IMMEDIATE (0-15 mins).
----------------------------------------------------------------------

--- INCIDENT #2 (Processed in 0.90 ms | Predictive: 0.89 ms) ---
[SITREP AUTO-GENERATED BY IBM LINUXONE AI]
INCIDENT CLASSIFICATION : INFRASTR

In [5]:
# 5. Latency & Hardware Acceleration Profiling
import numpy as np

latencies = []
for _ in range(100):
    t0 = time.time()
    _ = urgency_pipeline.predict(["Trapped victim under debris, need medical extraction immediately"])
    _ = category_pipeline.predict(["Trapped victim under debris, need medical extraction immediately"])
    latencies.append((time.time() - t0) * 1000)

print("=== IBM LinuxONE s390x Latency Benchmark ===")
print(f"Samples Tested         : 100")
print(f"Mean Inference Latency : {np.mean(latencies):.2f} ms")
print(f"P95 Inference Latency  : {np.percentile(latencies, 95):.2f} ms")
print(f"P99 Inference Latency  : {np.percentile(latencies, 99):.2f} ms")
print(f"Throughput             : {1000.0 / np.mean(latencies):.1f} predictions/sec/core")
print("\n*** PREREQUISITE LAB 2 COMPLETED SUCCESSFULLY ON IBM Z s390x ***")

=== IBM LinuxONE s390x Latency Benchmark ===
Samples Tested         : 100
Mean Inference Latency : 0.52 ms
P95 Inference Latency  : 0.61 ms
P99 Inference Latency  : 0.73 ms
Throughput             : 1933.0 predictions/sec/core

*** PREREQUISITE LAB 2 COMPLETED SUCCESSFULLY ON IBM Z s390x ***
